In [1]:
pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 23.2 MB/s eta 0:00:00


In [2]:
import os
import csv
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# Configuração do ChromeDriver
def iniciar_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    service = Service('chromedriver')  # Certifique-se de ter o chromedriver no caminho
    return webdriver.Chrome(service=service, options=options)

# Função para criar a pasta de saída
def criar_pasta_saida():
    if not os.path.exists('output'):
        os.mkdir('output')
        print("A pasta output foi criada.")

# Função para coletar produtos de uma categoria
def coletar_produtos(url, driver):
    produtos = []
    try:
        driver.get(url)
        time.sleep(3)  # Garantir que a página carregue completamente

        # Obter lista de produtos na página
        produtos_atual = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem'))
        )

        for produto in produtos_atual:
            try:
                # URL do Produto
                url_produto = produto.find_element(By.CSS_SELECTOR, 'a.vtex-product-summary-2-x-clearLink').get_attribute('href')

                # Imagem do Produto
                imagem_url = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-imageContainer img').get_attribute('src')

                # Nome do Produto
                nome_produto = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer').text.strip()

                # Preço Atual
                moeda = produto.find_element(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price').text.strip()

                # Código do Produto
                codigo_produto = produto.get_attribute('data-id') or "0"

                # Adicionar ao conjunto de produtos
                produtos.append([nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto])
            except Exception as e:
                print(f"Erro ao processar um produto: {e}")

        print(f"Coleta concluída para {url}. Total de produtos coletados: {len(produtos)}")

    except TimeoutException:
        print(f"Timeout ao carregar a página: {url}")
    except Exception as e:
        print(f"Erro ao coletar dados de {url}: {e}")

    return produtos

# Função para salvar dados em CSV
def salvar_csv(produtos, categoria):
    arquivo_saida = f'output/produtos_{categoria.replace("/", "_")}.csv'
    with open(arquivo_saida, mode='w', newline='', encoding='utf-8') as arquivo_csv:
        writer = csv.writer(arquivo_csv)
        writer.writerow(["Nome Empresa", "Data Coleta", "Código do Produto", "Imagem", "Nome do Produto", "Moeda", "URL do Produto"])
        writer.writerows(produtos)
    print(f"Dados salvos em {arquivo_saida}")

# Configurações iniciais
nome_empresa = "Joli"
data_coleta = datetime.now().strftime("%Y-%m-%d")
categorias = [
    "https://www.joli.com.br/chuveiros-e-aquecedores",
    "https://www.joli.com.br/cozinhas-e-areas-de-servico",
    "https://www.joli.com.br/banheiros-e-acessorios"
]

# Execução principal
if __name__ == "__main__":
    criar_pasta_saida()
    driver = iniciar_driver()

    try:
        for url_categoria in categorias:
            print(f"Iniciando a coleta de produtos da categoria: {url_categoria}")
            produtos = coletar_produtos(url_categoria, driver)
            if produtos:  # Somente salva se tiver produtos
                salvar_csv(produtos, url_categoria.split('/')[-1])
            else:
                print(f"Nenhum produto encontrado na categoria: {url_categoria}")
    finally:
        driver.quit()
        print("Execução finalizada.")


A pasta output foi criada.


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location
